# Broca: Bible — Knowledge Surgery Experiment

Train the **same** encoder/decoder architecture on the King James Bible.
Export the EAM, then combine with the Shakespeare EAM via HeatherDB algebra.

Same model. Same dims. Different knowledge. Then: **add, subtract, scale.**

In [ ]:
import math
import json
import urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
class EAMLayer(nn.Module):
    def __init__(self, num_locations=500, dim=64, k=20, beta=5.0, t_max=3, eta=0.01):
        super().__init__()
        self.num_locations = num_locations
        self.dim = dim
        self.k = k
        self.beta = beta
        self.t_max = t_max
        self.eta = eta

        addresses = F.normalize(torch.randn(num_locations, dim), dim=1)
        self.register_buffer("addresses", addresses)
        self.register_buffer("counters", torch.zeros(num_locations, dim))
        self.register_buffer("write_counts", torch.zeros(num_locations))

    def read(self, query):
        squeeze = query.dim() == 1
        if squeeze:
            query = query.unsqueeze(0)
        wc = self.write_counts.clamp(min=1e-12).unsqueeze(1)
        raw_patterns = self.counters / wc
        has_writes = self.write_counts > 0
        patterns = F.normalize(raw_patterns, dim=1) * has_writes.float().unsqueeze(1)
        xi = F.normalize(query, dim=1)
        k = min(self.k, has_writes.sum().item())
        if k == 0:
            return xi.squeeze(0) if squeeze else xi
        for _ in range(self.t_max):
            sims = torch.mm(xi, self.addresses.t())
            topk_sims, topk_idx = torch.topk(sims, k, dim=1)
            alpha = F.softmax(topk_sims * self.beta, dim=1)
            idx_expanded = topk_idx.unsqueeze(-1).expand(-1, -1, self.dim)
            topk_patterns = patterns.unsqueeze(0).expand(xi.shape[0], -1, -1)
            topk_patterns = torch.gather(topk_patterns, 1, idx_expanded)
            xi = (alpha.unsqueeze(-1) * topk_patterns).sum(dim=1)
            xi = F.normalize(xi, dim=1)
        return xi.squeeze(0) if squeeze else xi

    @torch.no_grad()
    def write(self, vector):
        if vector.dim() == 1:
            vector = vector.unsqueeze(0)
        vectors = F.normalize(vector, dim=1)
        k = min(self.k, self.num_locations)
        sims = torch.mm(vectors, self.addresses.t())
        topk_sims, topk_idx = torch.topk(sims, k, dim=1)
        weights = F.softmax(topk_sims * self.beta, dim=1)
        flat_idx = topk_idx.reshape(-1)
        flat_weights = weights.reshape(-1)
        vecs_expanded = vectors.unsqueeze(1).expand(-1, k, -1).reshape(-1, self.dim)
        weighted_vecs = flat_weights.unsqueeze(1) * vecs_expanded
        idx_for_counters = flat_idx.unsqueeze(1).expand(-1, self.dim)
        self.counters.scatter_add_(0, idx_for_counters, weighted_vecs)
        self.write_counts.scatter_add_(0, flat_idx, flat_weights)
        winners = topk_idx[:, 0]
        winner_addrs = self.addresses[winners]
        diff = vectors - winner_addrs
        self.addresses[winners] += self.eta * diff
        self.addresses[winners] = F.normalize(self.addresses[winners], dim=1)

    @torch.no_grad()
    def clear(self):
        self.counters.zero_()
        self.write_counts.zero_()

    def num_written(self):
        return (self.write_counts > 0).sum().item()

In [ ]:
class CharVocab:
    def __init__(self, text):
        chars = sorted(set(text))
        self.char_to_idx = {ch: i for i, ch in enumerate(chars)}
        self.idx_to_char = {i: ch for ch, i in self.char_to_idx.items()}
        self.vocab_size = len(chars)

    def encode(self, text):
        return [self.char_to_idx.get(ch, 0) for ch in text]

    def decode(self, indices):
        return "".join(self.idx_to_char.get(i, "?") for i in indices)


class ChunkDataset(Dataset):
    def __init__(self, text, vocab, context_length=128, chunk_size=32):
        self.context_length = context_length
        self.chunk_size = chunk_size
        self.data = torch.tensor(vocab.encode(text), dtype=torch.long)

    def __len__(self):
        return len(self.data) - self.context_length - self.chunk_size

    def __getitem__(self, idx):
        context = self.data[idx : idx + self.context_length]
        target = self.data[idx + self.context_length : idx + self.context_length + self.chunk_size]
        return context, target


def download_bible():
    """Download King James Bible from Project Gutenberg."""
    path = Path("kjv_bible.txt")
    if not path.exists():
        print("Downloading King James Bible...")
        url = "https://www.gutenberg.org/cache/epub/10/pg10.txt"
        text = urllib.request.urlopen(url).read().decode("utf-8")
        start = text.find("In the beginning")
        end = text.rfind("*** END OF THE PROJECT GUTENBERG EBOOK")
        bible = text[start:end].strip()
        path.write_text(bible)
        print(f"Saved {len(bible):,} characters")
    return path.read_text()

In [ ]:
class CharLM(nn.Module):
    def __init__(self, vocab_size=65, embed_dim=64, memory_dim=128,
                 context_length=128, chunk_size=32, dropout=0.2,
                 num_locations=2000, k=20, beta=5.0, t_max=3, eta=0.001):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.memory_dim = memory_dim
        self.context_length = context_length
        self.chunk_size = chunk_size

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embed_drop = nn.Dropout(dropout)
        self.encoder_gru = nn.GRU(embed_dim, memory_dim, num_layers=1,
                                   batch_first=True, dropout=0)
        self.decoder_gru = nn.GRU(embed_dim, memory_dim, num_layers=1,
                                   batch_first=True, dropout=0)
        self.out_drop = nn.Dropout(dropout)
        self.output = nn.Linear(memory_dim, vocab_size)
        self.memory = EAMLayer(num_locations=num_locations, dim=memory_dim,
                               k=k, beta=beta, t_max=t_max, eta=eta)

    def encode(self, char_indices):
        emb = self.embed_drop(self.embedding(char_indices))
        _, hidden = self.encoder_gru(emb)
        return F.normalize(hidden.squeeze(0), dim=-1)

    def decode_chunk(self, thought, seed_chars, target_chunk):
        decoder_input = torch.cat(
            [seed_chars.unsqueeze(1), target_chunk[:, :-1]], dim=1
        )
        emb = self.embed_drop(self.embedding(decoder_input))
        output, _ = self.decoder_gru(emb, thought.unsqueeze(0))
        return self.output(self.out_drop(output))

    @torch.no_grad()
    def generate_chunk(self, thought, seed_char, temperature=0.8):
        hidden = thought.unsqueeze(0)
        tokens = []
        current = seed_char
        for _ in range(self.chunk_size):
            emb = self.embedding(current).unsqueeze(1)
            out, hidden = self.decoder_gru(emb, hidden)
            logits = self.output(out.squeeze(1)) / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1).squeeze(1)
            tokens.append(next_token.item())
            current = next_token
        return tokens

    def forward(self, char_indices, target_chunk, use_memory=False):
        thought = self.encode(char_indices)
        if use_memory and self.memory.num_written() > 0:
            thought = self.memory.read(thought)
        seed_chars = char_indices[:, -1]
        return self.decode_chunk(thought, seed_chars, target_chunk)

    def freeze_controller(self):
        for param in self.embedding.parameters():
            param.requires_grad_(False)
        for param in self.encoder_gru.parameters():
            param.requires_grad_(False)
        for param in self.decoder_gru.parameters():
            param.requires_grad_(False)
        for param in self.output.parameters():
            param.requires_grad_(False)

## Hyperparameters
**Identical** to Shakespeare training — same dims, same chunk size, same EAM config.

In [ ]:
EMBED_DIM = 64
MEMORY_DIM = 128
CONTEXT_LENGTH = 128
CHUNK_SIZE = 32
DROPOUT = 0.2
NUM_LOCATIONS = 2000
K = 20
BETA = 5.0
T_MAX = 3
BATCH_SIZE = 256
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 7
TRAIN_SPLIT = 0.9
SEED = 42

## Load Bible

In [ ]:
torch.manual_seed(SEED)

text = download_bible()
vocab = CharVocab(text)
print(f"Corpus: {len(text):,} characters, {vocab.vocab_size} unique chars")
print(f"First 200 chars: {text[:200]}\n")

split_idx = int(len(text) * TRAIN_SPLIT)
train_ds = ChunkDataset(text[:split_idx], vocab, CONTEXT_LENGTH, CHUNK_SIZE)
val_ds = ChunkDataset(text[split_idx:], vocab, CONTEXT_LENGTH, CHUNK_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds):,} windows ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds):,} windows")
print(f"Chunk size: {CHUNK_SIZE} chars per thought")

## Build Model

In [ ]:
model = CharLM(
    vocab_size=vocab.vocab_size,
    embed_dim=EMBED_DIM,
    memory_dim=MEMORY_DIM,
    context_length=CONTEXT_LENGTH,
    chunk_size=CHUNK_SIZE,
    dropout=DROPOUT,
    num_locations=NUM_LOCATIONS,
    k=K, beta=BETA, t_max=T_MAX,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params:,} parameters")
print(f"  Embedding:    {sum(p.numel() for p in model.embedding.parameters()):,}")
print(f"  Encoder GRU:  {sum(p.numel() for p in model.encoder_gru.parameters()):,}")
print(f"  Decoder GRU:  {sum(p.numel() for p in model.decoder_gru.parameters()):,}")
print(f"  Output:       {sum(p.numel() for p in model.output.parameters()):,}")
print(f"  EAM:          {NUM_LOCATIONS} locations x {MEMORY_DIM}d")

## Phase 1: Train Encoder + Decoder

In [ ]:
@torch.no_grad()
def generate_sample(model, vocab, prompt="And God said", num_chunks=10,
                    temperature=0.8, use_memory=False):
    model.eval()
    context = vocab.encode(prompt)
    generated = list(context)
    for _ in range(num_chunks):
        window = generated[-model.context_length:]
        if len(window) < model.context_length:
            window = [0] * (model.context_length - len(window)) + window
        x = torch.tensor([window], dtype=torch.long, device=device)
        thought = model.encode(x)
        if use_memory and model.memory.num_written() > 0:
            thought = model.memory.read(thought)
        seed = torch.tensor([window[-1]], dtype=torch.long, device=device)
        chunk = model.generate_chunk(thought, seed, temperature)
        generated.extend(chunk)
    return vocab.decode(generated)


optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.CrossEntropyLoss()

best_val_ppl = float("inf")
best_state = None
patience_counter = 0

print("--- Phase 1: Training encoder + decoder on KJV Bible ---\n")

for epoch in range(EPOCHS):
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}", leave=False)
    for context, target_chunk in pbar:
        context, target_chunk = context.to(device), target_chunk.to(device)
        optimizer.zero_grad()
        logits = model(context, target_chunk, use_memory=False)
        loss = criterion(logits.reshape(-1, vocab.vocab_size), target_chunk.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * target_chunk.numel()
        total_correct += (logits.argmax(dim=-1) == target_chunk).sum().item()
        total_count += target_chunk.numel()
        pbar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{total_correct/total_count:.1%}")

    model.eval()
    val_loss, val_correct, val_count = 0.0, 0, 0
    with torch.no_grad():
        for context, target_chunk in val_loader:
            context, target_chunk = context.to(device), target_chunk.to(device)
            logits = model(context, target_chunk, use_memory=False)
            val_loss += criterion(
                logits.reshape(-1, vocab.vocab_size), target_chunk.reshape(-1)
            ).item() * target_chunk.numel()
            val_correct += (logits.argmax(dim=-1) == target_chunk).sum().item()
            val_count += target_chunk.numel()

    train_ppl = math.exp(total_loss / total_count)
    val_ppl = math.exp(val_loss / val_count)
    train_acc = total_correct / total_count
    val_acc = val_correct / val_count
    lr = optimizer.param_groups[0]["lr"]

    scheduler.step(val_ppl)

    if val_ppl < best_val_ppl:
        best_val_ppl = val_ppl
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        marker = " *"
    else:
        patience_counter += 1
        marker = ""

    print(f"Epoch {epoch+1:3d}: train_ppl={train_ppl:.2f} val_ppl={val_ppl:.2f} "
          f"train_acc={train_acc:.1%} val_acc={val_acc:.1%} lr={lr:.1e}{marker}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1} (best val_ppl={best_val_ppl:.2f})")
        break

model.load_state_dict(best_state)
model.to(device)
print(f"\nRestored best checkpoint (val_ppl={best_val_ppl:.2f})")

## Sample (no memory)

In [ ]:
print(generate_sample(model, vocab, prompt="And God said", use_memory=False))
print("\n---\n")
print(generate_sample(model, vocab, prompt="And the Lord spoke unto Moses", use_memory=False))

## Phase 2: Populate EAM

In [ ]:
model.freeze_controller()
model.eval()
model.memory.clear()

written = 0
with torch.no_grad():
    for context, _ in tqdm(train_loader, desc="Writing to EAM"):
        context = context.to(device)
        encoded = model.encode(context)
        model.memory.write(encoded)
        written += context.shape[0]

print(f"\nWrote {written:,} thought vectors to EAM")
print(f"Active locations: {model.memory.num_written()}/{NUM_LOCATIONS}")

## Evaluation: Direct vs Memory

In [ ]:
model.eval()
for label, use_mem in [("Direct", False), ("Memory", True)]:
    total_loss, total_correct, total_count = 0.0, 0, 0
    with torch.no_grad():
        for context, target_chunk in val_loader:
            context, target_chunk = context.to(device), target_chunk.to(device)
            logits = model(context, target_chunk, use_memory=use_mem)
            total_loss += criterion(
                logits.reshape(-1, vocab.vocab_size), target_chunk.reshape(-1)
            ).item() * target_chunk.numel()
            total_correct += (logits.argmax(dim=-1) == target_chunk).sum().item()
            total_count += target_chunk.numel()
    ppl = math.exp(total_loss / total_count)
    acc = total_correct / total_count
    print(f"{label:8s}: ppl={ppl:.2f}, acc={acc:.1%}")

## Sample (with EAM)

In [ ]:
print(generate_sample(model, vocab, prompt="And God said", use_memory=True))
print("\n---\n")
print(generate_sample(model, vocab, prompt="And the Lord spoke unto Moses", use_memory=True))
print("\n---\n")
print(generate_sample(model, vocab, prompt="In the beginning", use_memory=True))

## Export for HeatherDB

Downloads `broca_bible.json` + `broca_bible_model.pt`.

After downloading both this and the Shakespeare export:
```bash
# Import both
heather-fornix import -f broca.json       -c broca_shakespeare -d ./data
heather-fornix import -f broca_bible.json -c broca_bible       -d ./data

# Algebra
curl -X POST localhost:6380/algebra/add  -d '{"source_a":"broca_shakespeare","source_b":"broca_bible","target":"broca_blend"}'
curl -X POST localhost:6380/algebra/sub  -d '{"source_a":"broca_blend","source_b":"broca_bible","target":"broca_distilled"}'
curl -X POST localhost:6380/algebra/scale -d '{"source":"broca_bible","target":"broca_whisper","alpha":0.3}'
```

In [ ]:
model_cpu = model.cpu()
mem = model_cpu.memory

num_locs = mem.addresses.shape[0]
config = {
    "d": mem.dim, "l_0": num_locs, "l_max": num_locs * 2,
    "k": mem.k, "eta_0": mem.eta, "lambda": 0.9999,
    "eta_min": 0.001, "tau_split": 0.3, "tau_merge": 0.95,
    "gamma": 1.0, "tau_damp": 10.0, "tau_overload": 100.0,
    "beta": mem.beta, "t_max": mem.t_max, "epsilon": 1e-6,
}

addresses = mem.addresses.detach().double()
counters = mem.counters.detach().double()
write_counts = mem.write_counts.detach().double()

locations = [
    {"id": i, "address": addresses[i].tolist(),
     "counter": counters[i].tolist(),
     "write_count": float(write_counts[i])}
    for i in range(num_locs)
]

torch.save({
    "embedding": model_cpu.embedding.state_dict(),
    "encoder_gru": model_cpu.encoder_gru.state_dict(),
    "decoder_gru": model_cpu.decoder_gru.state_dict(),
    "output": model_cpu.output.state_dict(),
}, "broca_bible_model.pt")

export = {
    "config": config,
    "locations": locations,
    "controller_state": "broca_bible_model.pt",
    "prototypes": {},
    "vocab": {
        "char_to_idx": vocab.char_to_idx,
        "idx_to_char": {str(k): v for k, v in vocab.idx_to_char.items()},
    },
    "model_config": {
        "vocab_size": model_cpu.vocab_size,
        "embed_dim": model_cpu.embed_dim,
        "memory_dim": model_cpu.memory_dim,
        "context_length": model_cpu.context_length,
        "chunk_size": model_cpu.chunk_size,
    },
}

Path("broca_bible.json").write_text(json.dumps(export))
print(f"Exported: broca_bible.json ({Path('broca_bible.json').stat().st_size / 1024:.0f} KB)")
print(f"Exported: broca_bible_model.pt ({Path('broca_bible_model.pt').stat().st_size / 1024:.0f} KB)")
print(f"Locations: {num_locs}, Vocab: {vocab.vocab_size}, Chunk: {CHUNK_SIZE}")

In [ ]:
try:
    from google.colab import files
    files.download("broca_bible.json")
    files.download("broca_bible_model.pt")
except ImportError:
    print("Not in Colab — files saved locally.")